In [1]:
import papermill as pm
import numpy as np
import os
from pathlib import Path
BASE = Path.cwd()
# Optuna
#!pip install optuna
import optuna
from optuna.trial import create_trial

# GRID SEACH, define a parameter space and evaluate the simulation at each point uniformly

In [2]:
# temperature        = [4e-3, 5e-3, 6e-3, 8e-3]   # mK
# temperature_transv = [4e-3, 5e-3, 6e-3, 8e-3]   # mK
# tau_mixing         = [15, 20, 25, 30, 35] # s
# theta              = [10*np.pi/180, 15*np.pi/180, 20*np.pi/180, 25*np.pi/180] 
# print(temperature, temperature_transv)


# import multiprocessing as mp
# import papermill as pm

# def run_simulation(args):
#     t1, t2, tau, angle = args

#     output_name = f"Simulation_{t1*1e3:.2}mK_{t2*1e3:.2}mK"
#     print(f"\n>>> Executing {output_name}")

#     pm.execute_notebook(
#         "Simulation.ipynb",
#         f"output/notebooks/{output_notebook}_{bias}.ipynb",
#         parameters={
#             'Temperature'       : t1,
#             'Temperature_transv': t2,
#             'tau_mixing'        : tau,
#             'theta'             : angle,
#             'stringa'           : f"tau_{tau}s_theta_{int(angle*180/np.pi)}",
#             'bias'              : "0g"
#         }
#     )


# if __name__ == "__main__":
#     # genera tutte le combinazioni (equivalente ai due for annidati)
#     tasks = [(t1, t2, tau, angle) for t1 in temperature for t2 in temperature_transv for tau in tau_mixing for angle in theta]

#     # numero di processi (non saturare la macchina)
#     n_proc = min(len(tasks), max(1, mp.cpu_count() - 1))

#     with mp.Pool(processes= 7, maxtasksperchild=1) as pool:
#         pool.map(run_simulation, tasks)

# Bayesian optimization, smart search of the minimum.

In [3]:
# def objective(trial):
#     t1    = trial.suggest_float("Temperature", 0.5e-3, 10e-3)
#     t2    = trial.suggest_float("Temperature_transv", 0.5e-3, 10e-3)
#     tau   = trial.suggest_float("tau_mixing", 5, 100)
#     angle = trial.suggest_float("theta", 5*np.pi/180, 180*np.pi/180)

#     output_notebook = f"Simulation_tau_{tau:.2f}s_theta_{int(angle*180/np.pi)}_{t1*1e3:.2}mK_{t2*1e3:.2}mK"
#     output_name     = f"tau_{tau:.2f}s_theta_{int(angle*180/np.pi)}_axial_{t1*1e3:.2f}mK_transv_{t2*1e3:.2f}mK"
#     print(f"\n>>> Executing {output_name}")
    
#     pm.execute_notebook(
#         "Simulation.ipynb",
#         f"output/{output_notebook}.ipynb",
#         parameters={
#             'Temperature'       : t1,
#             'Temperature_transv': t2,
#             'tau_mixing'        : tau,
#             'theta'             : angle,
#             'stringa'           : output_name,
#             'bias'              : "0g"
#         }
#     )

#     data = np.load("output/" + output_name)
#     return float(data["metric"])

# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=200)

In [4]:
# print("Best LR:", study.best_value)
# print("Best params:", study.best_params)

# Simulation with comparison S-curve and Time Distributions
The objective function will perform a complete simulation, extracting simulated time distributions and comparing it to data time distributions. The objective function will also perform a simulation to extract the Scurve and compare it to the data. The metric will be the normalized LR of the time distributions 

In [5]:
import os

def append_to_dict(dictionary, key, item):
    if key in dictionary.keys():
        dictionary[key].append(item)
    else:
        dictionary.update({key : [item]})

os.makedirs("output",           exist_ok=True)
os.makedirs("output/figures",   exist_ok=True)
os.makedirs("output/notebooks", exist_ok=True)

list_biases = ['-0p75g', '0p0g', '0p5g', '0p75g', '-1p25g', '-0p37g', '0p25g', '1p0g', '-0p5g', '-0p25g']

## get older trials and load into a new study

In [16]:
from optuna.distributions import FloatDistribution, IntDistribution

In [17]:
files = os.listdir("output/")
#print(files)

outputs = ["output/" + item for item in files if "0p0g.npz" in item]
print("-")
print("LIST OF FILES")
print(outputs)

# Load in a Dictionary
simulation_list = {}
print("-"*20)
for output in outputs:
    print(output)
    filename   = output.split("/")[1]
    tau        = output.split("_")[1].replace("s","")
    theta      = output.split("_")[3]
    axial      = output.split("_")[5].replace("mK","")
    transverse = output.split("_")[7].replace("mK","")
    append_to_dict(simulation_list, output, float(axial))
    append_to_dict(simulation_list, output, float(transverse))
    append_to_dict(simulation_list, output, float(tau))
    append_to_dict(simulation_list, output, float(theta))

# # Recreate the Loss
# for key, value in simulation_list.items():
#     pm.execute_notebook(
#             "Metric_Worker.ipynb",
#             f"output/notebooks/Metric_Worker.ipynb",
#             parameters={
#                 'outputfile' : key.split("/")[1].replace("_0p0g.npz","")
#             },
#             cwd=str(BASE)
#         )

-
LIST OF FILES
['output/tau_28.58s_theta_129_axial_14.85mK_transv_11.65mK_0p0g.npz', 'output/tau_135.67s_theta_63_axial_16.29mK_transv_8.93mK_0p0g.npz', 'output/tau_138.45s_theta_31_axial_22.73mK_transv_9.98mK_0p0g.npz', 'output/tau_7.38s_theta_139_axial_14.16mK_transv_1.67mK_0p0g.npz', 'output/tau_109.48s_theta_94_axial_10.04mK_transv_2.09mK_0p0g.npz', 'output/tau_124.39s_theta_0_axial_21.14mK_transv_8.25mK_0p0g.npz', 'output/tau_66.32s_theta_97_axial_17.97mK_transv_20.62mK_0p0g.npz', 'output/tau_90.21s_theta_116_axial_1.38mK_transv_11.05mK_0p0g.npz', 'output/tau_86.51s_theta_84_axial_10.83mK_transv_3.11mK_0p0g.npz', 'output/tau_54.65s_theta_124_axial_5.92mK_transv_24.53mK_0p0g.npz', 'output/tau_95.26s_theta_40_axial_22.64mK_transv_14.60mK_0p0g.npz', 'output/tau_69.95s_theta_106_axial_24.46mK_transv_18.59mK_0p0g.npz', 'output/tau_37.98s_theta_37_axial_5.04mK_transv_22.19mK_0p0g.npz', 'output/tau_143.35s_theta_175_axial_0.51mK_transv_16.72mK_0p0g.npz', 'output/tau_126.12s_theta_44_axi

Executing:   0%|          | 0/18 [00:00<?, ?cell/s]

Executing:   0%|          | 0/18 [00:00<?, ?cell/s]

Executing:   0%|          | 0/18 [00:00<?, ?cell/s]

Executing:   0%|          | 0/18 [00:00<?, ?cell/s]

...

Executing:   0%|          | 0/18 [00:00<?, ?cell/s]

Executing:   0%|          | 0/18 [00:00<?, ?cell/s]

Executing:   0%|          | 0/18 [00:00<?, ?cell/s]

Executing:   0%|          | 0/18 [00:00<?, ?cell/s]

Executing:   0%|          | 0/18 [00:00<?, ?cell/s]

In [20]:
# Create Optuna Study
study = optuna.create_study(direction="minimize")

# recreate float distributions
distributions = {
    "t1": FloatDistribution(0.5e-3, 25e-3),
    "t2": FloatDistribution(0.5e-3, 25e-3),
    "tau_mixing": FloatDistribution(0, 150),
    "theta": FloatDistribution(0*np.pi/180, 180*np.pi/180)
}

# Recreate study
print("-"*20)
print("Recreating studies")
for key, value in simulation_list.items():
    # load the output
    data   = np.load(key)
    Chisq = data['Chisq_S']
    #comput the metric
    metric = Chisq[1] + Chisq[2]
    # retrieve the parameter of the trial
    params_dict = {"t1" : value[0]*1e-3, "t2" : value[1]*1e-3, "tau_mixing": value[2], "theta": value[3] * np.pi/180}
    
    print(key, "METRIC: %.2f" % metric, " t1 %.2f t2 %.2f" % (value[0], value[1]))
    trial = create_trial(
        params= params_dict,
        distributions=distributions,
        value= metric
    )

    

[I 2026-02-15 08:36:11,848] A new study created in memory with name: no-name-c590cc52-420e-4747-90a8-79939809e90f


--------------------
Recreating studies
output/tau_28.58s_theta_129_axial_14.85mK_transv_11.65mK_0p0g.npz METRIC: 123965.69  t1 14.85 t2 11.65
output/tau_135.67s_theta_63_axial_16.29mK_transv_8.93mK_0p0g.npz METRIC: 90032.89  t1 16.29 t2 8.93
output/tau_138.45s_theta_31_axial_22.73mK_transv_9.98mK_0p0g.npz METRIC: 98181.09  t1 22.73 t2 9.98
output/tau_7.38s_theta_139_axial_14.16mK_transv_1.67mK_0p0g.npz METRIC: 109770.29  t1 14.16 t2 1.67
output/tau_109.48s_theta_94_axial_10.04mK_transv_2.09mK_0p0g.npz METRIC: 95797.07  t1 10.04 t2 2.09
output/tau_124.39s_theta_0_axial_21.14mK_transv_8.25mK_0p0g.npz METRIC: 270.81  t1 21.14 t2 8.25
output/tau_66.32s_theta_97_axial_17.97mK_transv_20.62mK_0p0g.npz METRIC: 106391.59  t1 17.97 t2 20.62
output/tau_90.21s_theta_116_axial_1.38mK_transv_11.05mK_0p0g.npz METRIC: 105496.55  t1 1.38 t2 11.05
output/tau_86.51s_theta_84_axial_10.83mK_transv_3.11mK_0p0g.npz METRIC: 93679.20  t1 10.83 t2 3.11
output/tau_54.65s_theta_124_axial_5.92mK_transv_24.53mK_0p

In [21]:
def objective(trial):
    t1    = trial.suggest_float("t1", 0.5e-3, 25e-3)
    t2    = trial.suggest_float("t2", 0.5e-3, 25e-3)
    tau   = trial.suggest_float("tau_mixing", 0, 150)
    angle = trial.suggest_float("theta", 0*np.pi/180, 180*np.pi/180)
    #tau   = 100000
    #angle = 0
    
    output_notebook = f"Simulation_tau_{tau:.2f}s_theta_{int(angle*180/np.pi)}_{t1*1e3:.2}mK_{t2*1e3:.2}mK"
    output_name     = f"tau_{tau:.2f}s_theta_{int(angle*180/np.pi)}_axial_{t1*1e3:.2f}mK_transv_{t2*1e3:.2f}mK"
    print(f"\n>>> Executing {output_name}")

    for bias in list_biases:
        pm.execute_notebook(
            "Simulation.ipynb",
            f"output/notebooks/{output_notebook}_{bias}.ipynb",
            parameters={
                'Temperature'       : t1,
                'Temperature_transv': t2,
                'tau_mixing'        : tau,
                'theta'             : angle,
                'stringa'           : output_name,
                'bias'              : bias,
                'nAtoms'            : 3000,
            },
            cwd=str(BASE)
        )

    pm.execute_notebook(
            "Metric_Worker.ipynb",
            f"output/notebooks/Metric_Worker.ipynb",
            parameters={
                'outputfile' : output_name
            },
            cwd=str(BASE)
        )

    
    data = np.load("output/" + output_name + "_0p0g.npz")  # take the LR from the 0g files output.

    LR = data["metric"]
    Chisq = data['Chisq_S']

    print(f"Time Annihilation: {LR:.4f}, Scurve {Chisq}" )

    metric = Chisq[0] + Chisq[1]
    #return float(LR), float(Chi2sq[0]), float(Chisq[1]), float(Chisq[2]), float(Chisq[3])
    return metric

In [22]:
#study = optuna.create_study(directions=["minimize","minimize","minimize","minimize","minimize"])
#study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=200)


>>> Executing tau_125.00s_theta_55_axial_24.18mK_transv_11.91mK


Executing:   0%|          | 0/35 [00:00<?, ?cell/s]

Executing:   0%|          | 0/35 [00:00<?, ?cell/s]

Executing:   0%|          | 0/35 [00:00<?, ?cell/s]

Executing:   0%|          | 0/35 [00:00<?, ?cell/s]

...

Executing:   0%|          | 0/35 [00:00<?, ?cell/s]

Executing:   0%|          | 0/35 [00:00<?, ?cell/s]

Executing:   0%|          | 0/35 [00:00<?, ?cell/s]

[W 2026-02-16 10:20:03,076] Trial 65 failed with parameters: {'t1': 0.008341288700884731, 't2': 0.011753353802220387, 'tau_mixing': 98.64269906097383, 'theta': 0.36809499582158733} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/adriano/.local/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_56848/2578892657.py", line 14, in objective
    pm.execute_notebook(
  File "/home/adriano/.local/lib/python3.10/site-packages/papermill/execute.py", line 116, in execute_notebook
    nb = papermill_engines.execute_notebook_with_engine(
  File "/home/adriano/.local/lib/python3.10/site-packages/papermill/engines.py", line 48, in execute_notebook_with_engine
    return self.get_engine(engine_name).execute_notebook(nb, kernel_name, **kwargs)
  File "/home/adriano/.local/lib/python3.10/site-packages/papermill/engines.py", line 370, in execute_notebook
    cls.ex

KeyboardInterrupt: 